In [53]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.signal import butter, filtfilt, find_peaks

CSV_PATH = "/Users/Rodacki/Downloads/utt_participant_test_9f0cefcf_b7b_S1_20260424_143034.csv"

_FC_HZ = 1.0
_ORDER = 1
_DT = 1 / 60  # UTT ~60 Hz
_MIN_DIST_S = 0.15  # distância mínima entre picos/vales (s)
_PROM_FACTOR = 0.2  # prominence = fator * std(sinal após corte)


def interpolar_smartphone(time_array, signal_array, dt):
    Time_sp = np.asarray(time_array, dtype=float)
    Signal_sp = np.asarray(signal_array, dtype=float)

    # Remover NaNs (timestamps ou leituras ausentes)
    mask_valid = ~np.isnan(Time_sp) & ~np.isnan(Signal_sp)
    Time_sp = Time_sp[mask_valid]
    Signal_sp = Signal_sp[mask_valid]

    Time_diff_sp = np.diff(Time_sp)
    Time_diff_sp = np.insert(Time_diff_sp, 0, 0)
    Time_cumsum_sp = np.cumsum(Time_diff_sp)

    if not np.all(np.diff(Time_cumsum_sp) > 0):
        unique_indices_sp = np.unique(Time_cumsum_sp, return_index=True)[1]
        Time_cumsum_sp = Time_cumsum_sp[unique_indices_sp]
        Signal_sp = Signal_sp[unique_indices_sp]
        sorted_indices_sp = np.argsort(Time_cumsum_sp)
        Time_cumsum_sp = Time_cumsum_sp[sorted_indices_sp]
        Signal_sp = Signal_sp[sorted_indices_sp]

    New_Time_sp = np.arange(Time_cumsum_sp.min(), Time_cumsum_sp.max(), dt)
    cs_sp = CubicSpline(Time_cumsum_sp, Signal_sp)
    Signal_interpolated_sp = cs_sp(New_Time_sp)

    return Time_cumsum_sp, Signal_sp, New_Time_sp, Signal_interpolated_sp


df = pd.read_csv(CSV_PATH, comment="#")
t_s = df["row_t_ms"].astype(float) / 1000.0

# ay invertido e centrado (remove ~1g/DC)
ay = -df["ay"].to_numpy(dtype=float)
ay = ay - np.mean(ay)

# Interpolação igual ao prep_DP_UTT
_, _, t_u, ay_u = interpolar_smartphone(t_s, ay, _DT)

# Butterworth fc=1 Hz (passa-baixa) + filtfilt
fs_hz = 1.0 / _DT
b, a = butter(_ORDER, _FC_HZ, btype="low", fs=fs_hz)
ay_filt = filtfilt(b, a, ay_u)

# Remove os primeiros 1,3 s (eixo x relativo ao início do trecho mostrado)
_CROP_S = 1.3
_m = t_u >= _CROP_S
t_plot = t_u[_m] - _CROP_S
ay_plot = ay_filt[_m]

_min_dist = max(1, int(_MIN_DIST_S * fs_hz))
_prom = max(float(np.std(ay_plot)) * _PROM_FACTOR, 1e-9)
pk_i, _ = find_peaks(ay_plot, prominence=_prom, distance=15, height=0)
vl_i, _ = find_peaks(-ay_plot, prominence=_prom, distance=15, height=0)

_s = np.sign(ay_plot)
_s[_s == 0] = 1
_zc_edge = np.where(np.diff(_s))[0]
_i = _zc_edge
if len(_i) > 0:
    y0 = ay_plot[_i].astype(float)
    y1 = ay_plot[_i + 1].astype(float)
    t0 = t_plot[_i].astype(float)
    t1 = t_plot[_i + 1].astype(float)
    dn = y1 - y0
    m = dn != 0
    t_zc = (t0[m] - y0[m] * (t1[m] - t0[m]) / dn[m]).tolist()
else:
    t_zc = []


In [52]:
# Gráfico: sinal + picos + vales + zero crossings (executar a célula anterior antes)
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=ay_plot,
        mode="lines",
        name="sinal",
        line=dict(color="#ff7f0e", width=1),
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot[pk_i],
        y=ay_plot[pk_i],
        mode="markers",
        name="picos",
        marker=dict(symbol="triangle-up", size=10, color="#d62728", line=dict(width=0.5, color="white")),
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot[vl_i],
        y=ay_plot[vl_i],
        mode="markers",
        name="vales",
        marker=dict(symbol="triangle-down", size=10, color="#1f77b4", line=dict(width=0.5, color="white")),
    )
)
fig.add_trace(
    go.Scatter(
        x=t_zc,
        y=[0.0] * len(t_zc),
        mode="markers",
        name="zero crossings",
        marker=dict(symbol="x", size=9, color="#2ca02c", line=dict(width=1.5, color="#2ca02c")),
    )
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1, opacity=0.5)
fig.update_layout(
    title=(
        f"UTT — acel. Y + picos / vales / zero crossings "
        f"(Butter passa-baixa fc={_FC_HZ:g} Hz, ordem {_ORDER}, corte {_CROP_S:g} s)"
    ),
    xaxis_title="Tempo (s)",
    yaxis_title="ay (g, processado)",
    template="plotly_white",
    hovermode="x unified",
    height=450,
    margin=dict(l=50, r=20, t=50, b=50),
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()


In [ ]:
import numpy as np

# --- Separar ciclos: zc(up) > pico > zc(down) > vale ---
# Recalcula zero crossings com direção a partir de t_plot/ay_plot
_s = np.sign(ay_plot)
_s[_s == 0] = 1
_zc_edge = np.where(np.diff(_s) != 0)[0]

if len(_zc_edge) == 0:
    print("Nenhum zero crossing encontrado.")
else:
    y0 = ay_plot[_zc_edge].astype(float)
    y1 = ay_plot[_zc_edge + 1].astype(float)
    t0 = t_plot[_zc_edge].astype(float)
    t1 = t_plot[_zc_edge + 1].astype(float)
    dn = y1 - y0
    m = dn != 0

    t_zc_all = (t0[m] - y0[m] * (t1[m] - t0[m]) / dn[m])
    dir_up = (y0[m] < 0) & (y1[m] > 0)      # - -> +
    dir_down = (y0[m] > 0) & (y1[m] < 0)    # + -> -

    t_zc_up = np.asarray(t_zc_all[dir_up], dtype=float)
    t_zc_down = np.asarray(t_zc_all[dir_down], dtype=float)

    t_pk = np.asarray(t_plot[pk_i], dtype=float)
    t_vl = np.asarray(t_plot[vl_i], dtype=float)

    t_pk.sort()
    t_vl.sort()
    t_zc_up.sort()
    t_zc_down.sort()

    cycles = []  # cada ciclo: (t_start, t_mid, t_end)
    prev_end = -np.inf

    for tp in t_pk:
        # zc antes do pico (subida). Se não houver: 0.4 s antes do primeiro pico
        if t_zc_up.size > 0:
            i0 = np.searchsorted(t_zc_up, tp, side="left") - 1
            t_start = float(t_zc_up[i0]) if i0 >= 0 else float(tp - 0.4)
        else:
            t_start = float(tp - 0.4)

        # garante não sobrepor ciclos anteriores
        if t_start < prev_end:
            continue

        # zc depois do pico (descida)
        if t_zc_down.size == 0:
            continue
        i1 = np.searchsorted(t_zc_down, tp, side="right")
        if i1 >= t_zc_down.size:
            continue
        t_mid = float(t_zc_down[i1])

        # vale depois do zc(down)
        if t_vl.size == 0:
            continue
        i2 = np.searchsorted(t_vl, t_mid, side="right")
        if i2 >= t_vl.size:
            continue
        t_end = float(t_vl[i2])

        if not (t_start < tp < t_mid < t_end):
            continue

        cycles.append((t_start, t_mid, t_end))
        prev_end = t_end

    if len(cycles) == 0:
        print("Nenhum ciclo completo encontrado (zc>pico>zc>vale).")
    else:
        cycles = np.asarray(cycles, dtype=float)
        t_start = cycles[:, 0]
        t_mid = cycles[:, 1]
        t_end = cycles[:, 2]

        dur_cycle = t_end - t_start
        dur_rise = t_mid - t_start      # tempo positivo no ciclo
        dur_fall = t_end - t_mid        # tempo negativo no ciclo

        # tempo de transição: trechos fora de qualquer ciclo
        t_global0 = float(t_plot[0])
        t_global1 = float(t_plot[-1])

        gaps = []
        if t_start[0] > t_global0:
            gaps.append(t_start[0] - t_global0)
        for i in range(len(cycles) - 1):
            g = t_start[i + 1] - t_end[i]
            if g > 0:
                gaps.append(g)
        if t_global1 > t_end[-1]:
            gaps.append(t_global1 - t_end[-1])

        gaps = np.asarray(gaps, dtype=float)

        print(f"Número de ciclos: {len(cycles)}")
        print(f"Tempo médio de ciclo: {dur_cycle.mean():.4f} s")
        print(f"Tempo médio de subida (positivo no ciclo): {dur_rise.mean():.4f} s")
        print(f"Tempo médio de descida (negativo no ciclo): {dur_fall.mean():.4f} s")
        print(
            f"Tempo médio de transição (fora de ciclos): "
            f"{(gaps.mean() if gaps.size else 0.0):.4f} s"
        )
